In [140]:
%%capture

# # Package installs go here 
# %conda install polars
# %conda install plotly
# %conda install nbformat  # plotly needs it to draw charts in the notebook

# python imports 
import polars as pl
import plotly.io as pio
# VS Code draws the plotly mimetype natively (fast); the HTML copy keeps charts in the slides export
pio.renderers.default = "plotly_mimetype+notebook_connected"

## Data Processing

In [141]:
# Cleaning observations data 
observations = pl.read_csv("data/observation.csv")  
observations = observations.rename({
    "sd_ID": "survey_id", 
    "sd_SpeciesORIG": "species_id",  
    "sd_Breeding": "breeding_status",  
    "sd_quality" : "observation_quality", 
    "sd_Notes": "observation_notes", 
    "confidence_Level": "confidence_level"
}) 
observations

survey_id,species_id,breeding_status,observation_quality,observation_notes,confidence_level
i64,i64,i64,str,str,str
241376,58,0,"""NA""","""NA""","""NA"""
241376,137,0,"""NA""","""NA""","""NA"""
241376,129,0,"""NA""","""NA""","""NA"""
241376,153,0,"""NA""","""NA""","""NA"""
241376,124,0,"""NA""","""NA""","""NA"""
…,…,…,…,…,…
595767,982,0,"""NA""","""NA""","""NA"""
595751,500,0,"""NA""","""NA""","""NA"""
595769,654,0,"""NA""","""NA""","""NA"""


In [142]:
species = pl.read_csv("data/899_Species.csv")  
species = species.rename({
    "spNo": "species_id", 
    "sName": "species_name", 
    "sciName": "species_sciname", 
    "IOC_Species_ID": "IOC_species_id",  
    "observation_Count": "observation_count" 
})
species

BLTsort,numSort,species_id,species_name,species_sciname,IOC_species_id,species_Notes,observation_count
i64,i64,i64,str,str,str,str,i64
5,1,1,"""Emu""","""Dromaius novaehollandiae""","""31.0""","""Casuariiformes""",14361
3,2,2,"""Southern Cassowary""","""Casuarius casuarius""","""26.0""","""NA""",0
485,3,3,"""Southern Rockhopper Penguin""","""Eudyptes chrysocome""","""7333.0""","""NA""",0
488,4,4,"""Fiordland Penguin""","""Eudyptes pachyrhynchus""","""7335.0""","""Sphenisciformes""",16
490,5,5,"""Little Penguin""","""Eudyptula minor""","""7312.0""","""Sphenisciformes""",2625
…,…,…,…,…,…,…,…
1848,996,996,"""European Goldfinch""","""Carduelis carduelis""","""34692.0""","""Passeriformes""",13905
468,9966,9966,"""Red Billed Tropicbird""","""Phaethon aethereus""","""NA""","""NA""",19
1844,997,997,"""Common Greenfinch""","""Chloris chloris""","""34462.0""","""Passeriformes""",713


In [143]:
header = pl.read_csv(
    "data/header.csv",
    null_values=["NA", ""],
    infer_schema_length=100_000,
    schema_overrides={"k_Latitude": pl.Float64, "k_Longitude": pl.Float64},
)   

header = header.rename({
    "s_ID": "survey_id", 
    "s_StartF" : "start_date",
    "s_DateL" : "end_date",
    "s_Latitude" : "latitude",
    "s_Longitude" : "longitude",
    "s_Locality" : "locality",
    "s_Qualifier" : "qualifier",
    "s_Notes" : "notes", 
    "k_Latitude" : "k_latitude",
    "k_Longitude" : "k_longitude",
    "observation_Count" : "observation_count" 
}) 

header

survey_id,s_Atlasser,start_date,end_date,latitude,longitude,locality,qualifier,notes,s_DESref,s_AREVref,k_latitude,k_longitude,observation_count,confidence_Level,is_Collection
i64,i64,str,str,str,str,str,str,str,str,str,f64,f64,i64,i64,f64
3,23531,"""2005-04-09""","""2005-04-09""","""3255""","""15125""","""Freemans Waterhole""",null,"""ex C Driscoll""",null,null,3255.0,15125.0,1,0,1.0
4,23531,"""2005-04-09""","""2005-04-09""","""3345""","""15105""","""Turramurra""",null,"""ex T Wilson""",null,null,3345.0,15105.0,1,0,1.0
5,23531,"""2005-04-10""","""2005-04-10""","""3355""","""15115""","""Centennial Park""",null,"""ex K Hemming""",null,null,3355.0,15115.0,1,0,1.0
6,23531,"""2005-03-26""","""2005-03-26""","""3345""","""15105""","""East Killara""",null,"""Ex K Lindsay""",null,null,3345.0,15105.0,1,0,1.0
7,23531,"""2005-04-09""","""2005-04-09""","""2945""","""15255""","""Cowans Ponds""",null,"""Ex G Clancy""",null,null,2945.0,15255.0,1,0,1.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
595762,60003,"""2000-02-26""","""2000-03-04""","""3132""","""15904""","""Lord Howe Island""",null,null,"""BA2-3720""",null,3135.0,15905.0,30,2,null
595763,60003,"""2000-10-21""","""2000-10-22""","""3135""","""15905""","""Lord Howe Island""",null,null,"""BA2-5484""",null,3135.0,15905.0,31,0,null
595764,60003,"""2000-09-12""","""2000-11-22""","""3133""","""15905""","""Capella South Lord Howe Island""",null,null,"""BA2-5428""",null,3135.0,15905.0,1,1,1.0


In [144]:
full_observations = observations.join(header, left_on = "survey_id", right_on = "survey_id", how = "left") 
full_observations = full_observations.join(species, left_on = "species_id", right_on = "species_id", how = "left")  # joining full observations to get species names
full_observations.drop_in_place("observation_count_right") # Observation count is not needed once joined
full_observations


survey_id,species_id,breeding_status,observation_quality,observation_notes,confidence_level,s_Atlasser,start_date,end_date,latitude,longitude,locality,qualifier,notes,s_DESref,s_AREVref,k_latitude,k_longitude,observation_count,confidence_Level,is_Collection,BLTsort,numSort,species_name,species_sciname,IOC_species_id,species_Notes
i64,i64,i64,str,str,str,i64,str,str,str,str,str,str,str,str,str,f64,f64,i64,i64,f64,i64,i64,str,str,str,str
241376,58,0,"""NA""","""NA""","""NA""",23535,"""2015-10-25""","""2015-10-31""","""3135""","""15905""","""Lord Howe Island""",null,"""Ebird listings R Lloyd""",null,null,3135.0,15905.0,29,2,null,259,58,"""Purple Swamphen""","""Porphyrio porphyrio""","""NA""","""NA"""
241376,137,0,"""NA""","""NA""","""NA""",23535,"""2015-10-25""","""2015-10-31""","""3135""","""15905""","""Lord Howe Island""",null,"""Ebird listings R Lloyd""",null,null,3135.0,15905.0,29,2,null,288,137,"""Pacific Golden Plover""","""Pluvialis fulva""","""6454.0""","""Charadriiformes"""
241376,129,0,"""NA""","""NA""","""NA""",23535,"""2015-10-25""","""2015-10-31""","""3135""","""15905""","""Lord Howe Island""",null,"""Ebird listings R Lloyd""",null,null,3135.0,15905.0,29,2,null,337,129,"""Ruddy Turnstone""","""Arenaria interpres""","""6795.0""","""Charadriiformes"""
241376,153,0,"""NA""","""NA""","""NA""",23535,"""2015-10-25""","""2015-10-31""","""3135""","""15905""","""Lord Howe Island""",null,"""Ebird listings R Lloyd""",null,null,3135.0,15905.0,29,2,null,331,153,"""Bar-tailed Godwit""","""Limosa lapponica""","""6678.0""","""Charadriiformes"""
241376,124,0,"""NA""","""NA""","""NA""",23535,"""2015-10-25""","""2015-10-31""","""3135""","""15905""","""Lord Howe Island""",null,"""Ebird listings R Lloyd""",null,null,3135.0,15905.0,29,2,null,404,124,"""Black Noddy""","""Anous minutus""","""6931.0""","""Charadriiformes"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
595767,982,0,"""NA""","""NA""","""NA""",25592,"""2001-03-13""","""2001-03-13""","""3145""","""15915""","""Balls Pyramid""",null,null,"""BA2-9509""",null,3145.0,15915.0,8,0,null,406,982,"""Grey Noddy""","""Procelsterna albivitta""","""NA""","""NA"""
595751,500,0,"""NA""","""NA""","""NA""",60003,"""1999-03-08""","""1999-03-15""","""3132""","""15905""","""Lord Howe Island""",null,null,"""BA2-2383""",null,3135.0,15905.0,35,1,null,1338,500,"""Striated Fieldwren""","""Calamanthus fuliginosus""","""19576.0""","""Passeriformes"""
595769,654,0,"""NA""","""NA""","""NA""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1787,654,"""Painted Finch""","""Emblema pictum""","""33330.0""","""Passeriformes"""


In [145]:
locations = pl.read_csv(
    "data/location.csv",
    null_values=["NA", ""],
    infer_schema_length=100_000,
    schema_overrides={"k_Latitude": pl.Float64, "k_Longitude": pl.Float64},
)

locations

bl_ID,bl_Location,bl_Lat,bl_Lon,region_Code,header_Count,Min_Alt,Max_Alt
i64,str,i64,i64,str,i64,f64,f64
2903,"""Aarons Pass""",3255,14945,"""T""",111,550.0,1000.0
2904,"""Abbeywood""",3125,14835,"""WS""",137,200.0,300.0
2905,"""Abercombie""",3355,14915,"""WS""",23,350.0,1000.0
2906,"""Abercrombie Caves""",3355,14925,"""WS""",35,400.0,1100.0
2907,"""Aberfoyle Sugarloaf""",3015,15205,"""NTE""",43,450.0,1350.0
…,…,…,…,…,…,…,…
6258,null,3755,14955,null,1,null,null
6259,null,3755,15445,null,1,null,null
6260,null,3755,15545,null,1,null,null


In [146]:
traits = pl.read_csv(
    "data/traits.csv",
    null_values=["NA", ""],
    infer_schema_length=100_000,
)  

# Only 371 small birds have traits 
traits = traits.filter(pl.col("Body_Size") == "Small")
traits = traits.with_columns(
    pl.col("ID").cast(pl.Int64),
)

traits


ID,Scientific_Name,Common_Name,Order_Name,Family_Name,ALA_IUCN,Local_Status,Population,Species_Type,Primary_Diet,ALA_Diet_Code,Diet_Breadth,ESI,Foraging_Strategy,Primary_Habitat,Habitat_Breadth,Nest_Type,Nest_Substrate,Cooperative_Breeder,Incubating_Sex,Social_Behaviour,Monogamy,Mobility,Body_Mass_g,Body_Size,Exclusion_Zone,Elev_NormMin,Elev_NormMax,breeding_season_start,breeding_season_end,breeding_season_notes
i64,str,str,str,str,str,str,str,str,str,str,f64,f64,str,str,f64,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str
1614,"""Synoicus chinensis""","""Asian Blue Quail""","""Galliformes""","""Phasianidae""","""LC""","""Native""","""Australian""","""Bird""","""Omnivore""",null,3.0,0.92,null,"""Grassland""",4.0,"""Scrape""","""Ground""",null,"""Female""","""Pairs/Family""","""Mono""","""Various""",38.0,"""Small""",null,"""0""","""2300""",null,null,null
2237,"""Aegotheles cristatus""","""Australian Owlet-nightjar""","""Caprimulgiformes""","""Aegothelidae""","""LC""","""Native""","""Australian""","""Bird""","""Invertebrate""","""Invertebrates""",1.0,1.4,"""On ground""","""Woodland""",4.0,"""Cavity (Tree), No Nest""","""Building, Stump, Pole, Tree""",null,"""Both""",null,null,"""Sedentary""",48.0,"""Small""",null,"""0""","""1000""",null,null,null
2313,"""Collocalia esculenta""","""Glossy Swiftlet""","""Caprimulgiformes""","""Apodidae""","""LC""","""Native""","""Australian""","""Bird""","""Invertebrate""","""Invertebrates""",1.0,1.7,"""In air""","""Forest""",2.0,"""Half Cup/Shallow""","""Building, Rock, Tree""",null,"""Both""","""Various""",null,"""Sedentary""",8.66,"""Small""",null,"""0""","""3600""",null,null,null
2341,"""Collocalia natalis""","""Christmas Island Swiftlet""","""Caprimulgiformes""","""Apodidae""",null,null,null,"""Bird""","""Invertebrate""",null,1.0,1.7,null,"""Forest""",2.0,"""Cup/Bowl""","""Grass""",null,"""Both""","""Territorial""",null,"""Sedentary""",4.75,"""Small""",null,null,null,null,null,null
2376,"""Aerodramus terraereginae""","""Australian Swiftlet""","""Caprimulgiformes""","""Apodidae""","""LC""","""Native""","""Australian""","""Bird""","""Invertebrate""","""Invertebrates""",1.0,1.52,"""In air""","""Savanna""",3.0,"""Cavity (Tree)""","""Rock""",null,"""Both""","""Territorial""",null,"""Sedentary""",11.2,"""Small""",null,"""0""","""500""",null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
34462,"""Chloris chloris""","""European Greenfinch""","""Passeriformes""","""Fringillidae""","""LC""","""Exotic""","""Introduced""","""Bird""","""Herbivore""",null,4.0,0.8,null,"""Forest""",4.0,"""Cup/Bowl""","""Shrub/Bush/Vine, Tree, Nest, B…",null,"""Female""","""Various""","""Mono""","""Various""",27.72,"""Small""",null,"""0""","""1400""",null,null,null
34641,"""Acanthis flammea""","""Redpoll""","""Passeriformes""","""Fringillidae""","""LC""","""Exotic""","""Introduced""","""Bird""","""Seed""","""Seed""",3.0,0.48,"""On ground""","""Plains""",11.0,"""Cup/Bowl""","""Tree""","""Occasional help""","""Female""","""Various""","""Mono""","""Various""",15.47,"""Small""",null,"""0""","""2100""",null,null,null
34692,"""Carduelis carduelis""","""European Goldfinch""","""Passeriformes""","""Fringillidae""","""LC""","""Exotic""","""Introduced""","""Bird""","""Seed""","""Seed""",4.0,0.44,"""Understorey""","""Woodland""",9.0,"""Cup/Bowl""","""Tree, Shrub/Bush/Vine""","""Solitary breeder""","""Female""","""Various""","""Mono""","""Various""",19.0,"""Small""",null,"""0""","""2300""",null,null,null


In [147]:
species_with_traits =traits.join(species, left_on="ID", right_on = "species_id", how = "left") 
species_with_traits.describe()

statistic,ID,Scientific_Name,Common_Name,Order_Name,Family_Name,ALA_IUCN,Local_Status,Population,Species_Type,Primary_Diet,ALA_Diet_Code,Diet_Breadth,ESI,Foraging_Strategy,Primary_Habitat,Habitat_Breadth,Nest_Type,Nest_Substrate,Cooperative_Breeder,Incubating_Sex,Social_Behaviour,Monogamy,Mobility,Body_Mass_g,Body_Size,Exclusion_Zone,Elev_NormMin,Elev_NormMax,breeding_season_start,breeding_season_end,breeding_season_notes,BLTsort,numSort,species_name,species_sciname,IOC_species_id,species_Notes,observation_count
str,f64,str,str,str,str,str,str,str,str,str,str,f64,f64,str,str,f64,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,f64,f64,str,str,str,str,f64
"""count""",371.0,"""371""","""371""","""371""","""370""","""364""","""351""","""351""","""371""","""371""","""300""",371.0,371.0,"""300""","""371""",371.0,"""354""","""344""","""84""","""246""","""242""","""113""","""370""",371.0,"""371""","""0""","""349""","""348""","""0""","""0""","""0""",0.0,0.0,"""0""","""0""","""0""","""0""",0.0
"""null_count""",0.0,"""0""","""0""","""0""","""1""","""7""","""20""","""20""","""0""","""0""","""71""",0.0,0.0,"""71""","""0""",0.0,"""17""","""27""","""287""","""125""","""129""","""258""","""1""",0.0,"""0""","""371""","""22""","""23""","""371""","""371""","""371""",371.0,371.0,"""371""","""371""","""371""","""371""",371.0
"""mean""",20547.536388,null,null,null,null,null,null,null,null,null,null,1.897574,1.250889,null,null,3.80593,null,null,null,null,null,null,null,21.189569,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""std""",7654.735904,null,null,null,null,null,null,null,null,null,null,0.99744,0.336795,null,null,1.750696,null,null,null,null,null,null,null,11.545705,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""min""",1614.0,"""Acanthis flammea""","""Amur Stonechat""","""Caprimulgiformes""","""Acanthizidae""","""CR""","""Exotic""","""Australian""","""Bird""","""Carnivore""","""Fish""",1.0,0.44,"""Canopy""","""Artificial""",1.0,"""Burrow""","""Bamboo, Building, Ground, Shru…","""Cooperative breeder""","""B,F""","""Pairs/Family""","""Mono""","""Migratory""",4.75,"""Small""",null,"""0""","""0""",null,null,null,null,null,null,null,null,null,null
"""25%""",18895.0,null,null,null,null,null,null,null,null,null,null,1.0,1.0,null,null,3.0,null,null,null,null,null,null,null,11.75,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""50%""",19693.0,null,null,null,null,null,null,null,null,null,null,2.0,1.22,null,null,4.0,null,null,null,null,null,null,null,18.06,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""75%""",23467.0,null,null,null,null,null,null,null,null,null,null,2.0,1.52,null,null,5.0,null,null,null,null,null,null,null,30.25,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""max""",35001.0,"""Zosterops tenuirostris""","""Zitting Cisticola""","""Psittaciformes""","""Zosteropidae""","""VU""","""Vagrant""","""Vagrant (from NZ introduction)""","""Bird""","""Seed""","""Seed""",5.0,2.0,"""Water surface""","""Woodland""",11.0,"""Sphere/Globular, Dome/Oven""","""Tree, Shrub/Bush/Vine, Rock""","""Solitary breeder""","""Male""","""Various""","""Poly""","""Various""",49.25,"""Small""",null,"""L""","""M""",null,null,null,null,null,null,null,null,null,null


In [148]:

synonyms = pl.read_csv("data/synonyms.csv") 
synonyms

ioc_15_1,english_name_birdbase,latin_name_birdbase,hbw_birdlife_v9_1,ioc_world_bird_list_v15_1,ebird_clements_v2024b,avilist_v1_2025
i64,str,str,str,str,str,str
1,"""Common Ostrich""","""Struthio camelus""","""Struthio camelus""","""Struthio camelus""","""Struthio camelus""","""Struthio camelus"""
26,"""Southern Cassowary""","""Casuarius casuarius""","""Casuarius casuarius""","""Casuarius casuarius""","""Casuarius casuarius""","""Casuarius casuarius"""
31,"""Common Emu""","""Dromaius novaehollandiae""","""Dromaius novaehollandiae""","""Dromaius novaehollandiae""","""Dromaius novaehollandiae""","""Dromaius novaehollandiae"""
227,"""Magpie Goose""","""Anseranas semipalmata""","""Anseranas semipalmata""","""Anseranas semipalmata""","""Anseranas semipalmata""","""Anseranas semipalmata"""
232,"""Spotted Whistling-duck""","""Dendrocygna guttata""","""Dendrocygna guttata""","""Dendrocygna guttata""","""Dendrocygna guttata""","""Dendrocygna guttata"""
…,…,…,…,…,…,…
354,"""Muscovy Duck""","""Cairina moschata""","""Cairina moschata""","""Cairina moschata""","""Cairina moschata""","""Cairina moschata"""
33681,"""Blue-breasted Cordon-bleu""","""Uraeginthus angolensis""","""Uraeginthus angolensis""","""Uraeginthus angolensis""","""Uraeginthus angolensis""","""Uraeginthus angolensis"""
14203,"""Rosy-faced Lovebird""","""Agapornis roseicollis""","""Agapornis roseicollis""","""Agapornis roseicollis""","""Agapornis roseicollis""","""Agapornis roseicollis"""


In [149]:


# Reading DES data: only the first sheet of the spreadsheet
# engine="openpyxl" because polars' default Excel reader (fastexcel) isn't installed
des = pl.read_excel(
    "data/des.xlsx",
    sheet_name="DES_Backlog_Format",
    engine="openpyxl",
)
des

Survey ID,DES Ref,Latitude,Longitude,Observation Start Date,Observation End Date,Locality,Comments,Collection flag,Observer Id,Observer Last Name,Observer First Name,Location name,Region code,Species Common Name,Species ID,Breeding Code,Quality flag,Species Comments [Notes],K-Latitude,K-Longitude
i64,str,i64,i64,datetime[μs],datetime[μs],str,str,str,str,str,str,str,str,str,str,str,null,str,str,str
900000,"""BPC46-26009.028""",2945,15035,2025-08-01 00:00:00,2025-08-31 00:00:00,"""Mount Rodd""","""NULL""","""NULL""","""26009""","""Swain""","""Bob""","""Mount Rodd""","""WS""","""Emu""","""1""","""Y""",null,"""NULL""","""2945""","""15035"""
900000,"""BPC46-26009.028""",2945,15035,2025-08-01 00:00:00,2025-08-31 00:00:00,"""Mount Rodd""","""NULL""","""NULL""","""26009""","""Swain""","""Bob""","""Mount Rodd""","""WS""","""Masked Lapwing""","""133""","""Y""",null,"""NULL""","""2945""","""15035"""
900000,"""BPC46-26009.028""",2945,15035,2025-08-01 00:00:00,2025-08-31 00:00:00,"""Mount Rodd""","""NULL""","""NULL""","""26009""","""Swain""","""Bob""","""Mount Rodd""","""WS""","""Australian Wood Duck""","""202""",null,null,"""NULL""","""2945""","""15035"""
900000,"""BPC46-26009.028""",2945,15035,2025-08-01 00:00:00,2025-08-31 00:00:00,"""Mount Rodd""","""NULL""","""NULL""","""26009""","""Swain""","""Bob""","""Mount Rodd""","""WS""","""Pacific Black Duck""","""208""",null,null,"""NULL""","""2945""","""15035"""
900000,"""BPC46-26009.028""",2945,15035,2025-08-01 00:00:00,2025-08-31 00:00:00,"""Mount Rodd""","""NULL""","""NULL""","""26009""","""Swain""","""Bob""","""Mount Rodd""","""WS""","""Australasian Grebe""","""61""",null,null,"""NULL""","""2945""","""15035"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
902577,"""BPC18-26044.033""",3155,14925,2022-08-21 00:00:00,2022-08-21 00:00:00,"""Moggymill""","""NULL""","""NULL""","""26044""","""Goulder""","""Graham""","""Moggymill""","""WS""","""Stubble Quail""","""9""",null,null,"""NULL""","""3155""","""14925"""
902577,"""BPC18-26044.033""",3155,14925,2022-08-21 00:00:00,2022-08-21 00:00:00,"""Moggymill""","""NULL""","""NULL""","""26044""","""Goulder""","""Graham""","""Moggymill""","""WS""","""Australian Magpie""","""705""",null,null,"""NULL""","""3155""","""14925"""
902577,"""BPC18-26044.033""",3155,14925,2022-08-21 00:00:00,2022-08-21 00:00:00,"""Moggymill""","""NULL""","""NULL""","""26044""","""Goulder""","""Graham""","""Moggymill""","""WS""","""Willie Wagtail""","""364""",null,null,"""NULL""","""3155""","""14925"""


In [150]:

# Reading DES data: only the first sheet of the spreadsheet
# engine="openpyxl" because polars' default Excel reader (fastexcel) isn't installed
des = pl.read_excel(
    "data/des.xlsx",
    sheet_name="DES_Backlog_Format",
    engine="openpyxl",
)
des

Survey ID,DES Ref,Latitude,Longitude,Observation Start Date,Observation End Date,Locality,Comments,Collection flag,Observer Id,Observer Last Name,Observer First Name,Location name,Region code,Species Common Name,Species ID,Breeding Code,Quality flag,Species Comments [Notes],K-Latitude,K-Longitude
i64,str,i64,i64,datetime[μs],datetime[μs],str,str,str,str,str,str,str,str,str,str,str,null,str,str,str
900000,"""BPC46-26009.028""",2945,15035,2025-08-01 00:00:00,2025-08-31 00:00:00,"""Mount Rodd""","""NULL""","""NULL""","""26009""","""Swain""","""Bob""","""Mount Rodd""","""WS""","""Emu""","""1""","""Y""",null,"""NULL""","""2945""","""15035"""
900000,"""BPC46-26009.028""",2945,15035,2025-08-01 00:00:00,2025-08-31 00:00:00,"""Mount Rodd""","""NULL""","""NULL""","""26009""","""Swain""","""Bob""","""Mount Rodd""","""WS""","""Masked Lapwing""","""133""","""Y""",null,"""NULL""","""2945""","""15035"""
900000,"""BPC46-26009.028""",2945,15035,2025-08-01 00:00:00,2025-08-31 00:00:00,"""Mount Rodd""","""NULL""","""NULL""","""26009""","""Swain""","""Bob""","""Mount Rodd""","""WS""","""Australian Wood Duck""","""202""",null,null,"""NULL""","""2945""","""15035"""
900000,"""BPC46-26009.028""",2945,15035,2025-08-01 00:00:00,2025-08-31 00:00:00,"""Mount Rodd""","""NULL""","""NULL""","""26009""","""Swain""","""Bob""","""Mount Rodd""","""WS""","""Pacific Black Duck""","""208""",null,null,"""NULL""","""2945""","""15035"""
900000,"""BPC46-26009.028""",2945,15035,2025-08-01 00:00:00,2025-08-31 00:00:00,"""Mount Rodd""","""NULL""","""NULL""","""26009""","""Swain""","""Bob""","""Mount Rodd""","""WS""","""Australasian Grebe""","""61""",null,null,"""NULL""","""2945""","""15035"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
902577,"""BPC18-26044.033""",3155,14925,2022-08-21 00:00:00,2022-08-21 00:00:00,"""Moggymill""","""NULL""","""NULL""","""26044""","""Goulder""","""Graham""","""Moggymill""","""WS""","""Stubble Quail""","""9""",null,null,"""NULL""","""3155""","""14925"""
902577,"""BPC18-26044.033""",3155,14925,2022-08-21 00:00:00,2022-08-21 00:00:00,"""Moggymill""","""NULL""","""NULL""","""26044""","""Goulder""","""Graham""","""Moggymill""","""WS""","""Australian Magpie""","""705""",null,null,"""NULL""","""3155""","""14925"""
902577,"""BPC18-26044.033""",3155,14925,2022-08-21 00:00:00,2022-08-21 00:00:00,"""Moggymill""","""NULL""","""NULL""","""26044""","""Goulder""","""Graham""","""Moggymill""","""WS""","""Willie Wagtail""","""364""",null,null,"""NULL""","""3155""","""14925"""


In [151]:
# Helper function that converts form degree:minutes to degree:decimal
def ddmm_to_dd(column_name: str) -> pl.Expr:
    degree = pl.col(column_name) // 100
    minutes = pl.col(column_name) - degree * 100
    return degree + (minutes / 60)

In [152]:
# Converts DES coordinates from degrees-minutes (DDMM) to decimal degrees
# The K- columns are a cleaned copy of Latitude/Longitude: they fix small typos
# (e.g. 15034 -> 15035) and hold the text "NULL" where the original was unusable.
des = des.rename({
    "K-Latitude": "key_latitude", "K-Longitude": "key_longitude",
    "Latitude": "latitude", "Longitude": "longitude",
})

# Turn the text "NULL" into a real missing value, then make the key columns numbers
des = des.with_columns(
    pl.col("key_latitude", "key_longitude").replace("NULL", None).cast(pl.Int64)
)

# A DDMM value is only a real coordinate if its minutes are below 60 and its degrees
# are inside Australia's range. Anything else (e.g. 30015 or 15275) becomes missing.
def valid_ddmm(column_name: str, min_degree: int, max_degree: int) -> pl.Expr:
    degree = pl.col(column_name) // 100
    minutes = pl.col(column_name) % 100
    return (minutes < 60) & degree.is_between(min_degree, max_degree)

des = des.with_columns(
    pl.when(valid_ddmm(c, 9, 45)).then(pl.col(c)).otherwise(None).alias(c)
    for c in ["latitude", "key_latitude"]
).with_columns(
    pl.when(valid_ddmm(c, 112, 160)).then(pl.col(c)).otherwise(None).alias(c)
    for c in ["longitude", "key_longitude"]
)

# Same conversion as for eBird: latitude is south, so it becomes negative
des = des.with_columns(
    key_latitude=-ddmm_to_dd("key_latitude"),
    key_longitude=ddmm_to_dd("key_longitude"),
    latitude=-ddmm_to_dd("latitude"),
    longitude=ddmm_to_dd("longitude"),
)

des.select("latitude", "longitude", "key_latitude", "key_longitude").describe()

statistic,latitude,longitude,key_latitude,key_longitude
str,f64,f64,f64,f64
"""count""",104103.0,104125.0,104103.0,104125.0
"""null_count""",37.0,15.0,37.0,15.0
"""mean""",-31.436353,151.045275,-31.436357,151.045286
"""std""",1.804477,2.290407,1.804481,2.290405
"""min""",-37.416667,141.083333,-37.416667,141.083333
"""25%""",-33.25,149.916667,-33.25,149.916667
"""50%""",-31.083333,151.416667,-31.083333,151.416667
"""75%""",-30.083333,152.916667,-30.083333,152.916667
"""max""",-21.083333,153.583333,-21.083333,153.583333


In [153]:
# Number of regions in the dataset - 9 Regions  

des_sighting_locations = des['Region code'].value_counts(sort = True)  


region_names = {"NC": "Northern Coast", 
                "RP": "Riverine Plane", 
                "A": "Arid", 
                "CSC": "Central and South Coast", 
                "WS": "Western Slopes", 
                "T": "Tablelands", 
                "MAW": "Malee and Acacia Woodlands", 
                "CP": "Central Peneplain",
                "TS": "Tasman Sea"
                }

In [154]:
import plotly.express as px
# NSW extent (same bounds as the R notebook)
NSW_LAT = (-37.7, -28.0)
NSW_LON = (140.5, 164.0)

# eBird key coordinates are the centres of 10-minute Bird Atlas grid cells,
# so each tile reaches 5 minutes either side of its centre
HALF_CELL = 5 / 60

def square(lat, lon):
    """GeoJSON polygon for the grid cell centred on (lat, lon)."""
    h = HALF_CELL
    return [[[lon - h, lat - h], [lon + h, lat - h],
             [lon + h, lat + h], [lon - h, lat + h], [lon - h, lat - h]]]

# # DES heatmap: number of observations in each 10-minute grid cell across NSW

# # Keep only locations inside NSW (NSW_LAT and NSW_LON are defined in the eBird heatmap cell)
# des_nsw = des.filter(
#     pl.col("key_latitude").is_between(*NSW_LAT),
#     pl.col("key_longitude").is_between(*NSW_LON),
# )

# # Frequency table of locations, with an id for each grid cell
# location_counts = (
#     des_nsw
#     .select("key_latitude", "key_longitude")
#     .drop_nulls()
#     .group_by("key_latitude", "key_longitude")
#     .len("observations")
#     # len() gives an unsigned integer, which plotly mistakes for categories
#     # (one layer per distinct count). Casting to Int64 keeps it numeric.
#     .with_columns(pl.col("observations").cast(pl.Int64))
#     .with_row_index("cell_id")
# )

# # Put each cell into a logarithmic band. left_closed=True puts exactly 10 in "10–99", not "1–9".
# BAND_LABELS = ["1–9", "10–99", "100–999", "1,000–9,999", "10,000+"]
# location_counts = location_counts.with_columns(
#     band=pl.col("observations").cut(
#         breaks=[10, 100, 1000, 10000], labels=BAND_LABELS, left_closed=True
#     )
# )

# # Each key coordinate is the centre of a 10-minute square, so h is half its width
# h = 5 / 60

# def square(lat, lon):
#     return [[[lon - h, lat - h], [lon + h, lat - h],
#              [lon + h, lat + h], [lon - h, lat + h], [lon - h, lat - h]]]

# shapes = {
#     "type": "FeatureCollection",
#     "features": [
#         {"type": "Feature", "id": row["cell_id"],
#          "geometry": {"type": "Polygon", "coordinates": square(row["key_latitude"], row["key_longitude"])}}
#         for row in location_counts.iter_rows(named=True)
#     ],
# }

# band_colours = dict(zip(BAND_LABELS, px.colors.sample_colorscale("Plasma_r", 5)))

# fig = px.choropleth_map(
#     location_counts, geojson=shapes, locations="cell_id", color="band",
#     color_discrete_map=band_colours,
#     category_orders={"band": BAND_LABELS},   # legend runs from fewest to most
#     custom_data=["observations"],
#     map_style="open-street-map", center={"lat": -32.8, "lon": 147.0}, zoom=4.6,
#     opacity=0.85,
#     title="Heatmap of DES observations in NSW: 2020 - 2026",
# )
# fig.update_traces(marker_line_width=0, hovertemplate="%{customdata[0]:,} observations<extra></extra>")
# fig.update_layout(height=650, margin=dict(l=0, r=0, t=50, b=0), legend_title_text="Observations")
# fig.show()

## Observations heatmap
The observations table has no coordinates, so each sighting takes its location from the survey it belongs to (`survey_id` in observations matches `survey_id` in the survey headers).

In [155]:
# Observations heatmap: number of sightings in each 10-minute grid cell across NSW

# Attach each sighting to its survey to get the survey's grid-cell coordinates.
# An inner join drops the ~1,200 sightings whose survey isn't in the header table.
obs_with_location = (
    observations
    .join(header.select("survey_id", "k_latitude", "k_longitude"), on="survey_id")
    .rename({"k_latitude": "key_latitude", "k_longitude": "key_longitude"})
    # Header coordinates are DDMM like eBird's, and latitude is south
    .with_columns(
        key_latitude=-ddmm_to_dd("key_latitude"),
        key_longitude=ddmm_to_dd("key_longitude"),
    )
)

# Count sightings per grid cell inside NSW (a handful of typo'd cells fall outside and drop out here)
obs_counts = (
    obs_with_location
    .filter(
        pl.col("key_latitude").is_between(*NSW_LAT),
        pl.col("key_longitude").is_between(*NSW_LON),
    )
    .group_by("key_latitude", "key_longitude")
    .len("observations")
    .with_columns(pl.col("observations").cast(pl.Int64))   # keep plotly treating it as numeric
    .with_row_index("cell_id")
)

# Busiest cells pass 100,000 sightings, so this needs two more bands than the DES map
OBS_BAND_LABELS = ["1–9", "10–99", "100–999", "1,000–9,999", "10,000–99,999", "100,000+"]
obs_counts = obs_counts.with_columns(
    band=pl.col("observations").cut(
        breaks=[10, 100, 1000, 10000, 100000], labels=OBS_BAND_LABELS, left_closed=True
    )
)

# Grid-cell squares, using square() from the cell above
obs_shapes = {
    "type": "FeatureCollection",
    "features": [
        {"type": "Feature", "id": row["cell_id"],
         "geometry": {"type": "Polygon", "coordinates": square(row["key_latitude"], row["key_longitude"])}}
        for row in obs_counts.iter_rows(named=True)
    ],
}

obs_band_colours = dict(zip(OBS_BAND_LABELS, px.colors.sample_colorscale("Plasma_r", len(OBS_BAND_LABELS))))

fig = px.choropleth_map(
    obs_counts, geojson=obs_shapes, locations="cell_id", color="band",
    color_discrete_map=obs_band_colours,
    category_orders={"band": OBS_BAND_LABELS},
    custom_data=["observations"],
    map_style="open-street-map", center={"lat": -32.8, "lon": 147.0}, zoom=4.6,
    opacity=0.85,
    title="Heatmap of observations in NSW from 1960 - 2019",
)
fig.update_traces(marker_line_width=0, hovertemplate="%{customdata[0]:,} observations<extra></extra>")
fig.update_layout(height=650, margin=dict(l=0, r=0, t=50, b=0), legend_title_text="Observations")
fig.show()

/var/folders/st/x242pp4d5qx9rg34ddcl8nb40000gn/T/ipykernel_10284/3286595672.py:32: DeprecationWarning: `cut` is deprecated; use `bin_intervals` instead
  band=pl.col("observations").cut(


In [156]:
endangered_species = traits.filter(pl.col("ALA_IUCN").str.contains("EN"))  
endangered_species

ID,Scientific_Name,Common_Name,Order_Name,Family_Name,ALA_IUCN,Local_Status,Population,Species_Type,Primary_Diet,ALA_Diet_Code,Diet_Breadth,ESI,Foraging_Strategy,Primary_Habitat,Habitat_Breadth,Nest_Type,Nest_Substrate,Cooperative_Breeder,Incubating_Sex,Social_Behaviour,Monogamy,Mobility,Body_Mass_g,Body_Size,Exclusion_Zone,Elev_NormMin,Elev_NormMax,breeding_season_start,breeding_season_end,breeding_season_notes
i64,str,str,str,str,str,str,str,str,str,str,f64,f64,str,str,f64,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str
13844,"""Psephotellus chrysopterygius""","""Golden-shouldered Parrot""","""Psittaciformes""","""Psittaculidae""","""EN""","""Native""","""Endemic""","""Bird""","""Seed""",null,1.0,1.52,null,"""Forest""",3.0,"""Burrow""","""Ground, Nest""",null,"""Female""","""Social""","""Mono""","""Various""",42.05,"""Small""",null,"""0""","""150""",null,null,null
18691,"""Atrichornis rufescens""","""Rufous Scrub-bird""","""Passeriformes""","""Atrichornithidae""","""EN""","""Native""","""Endemic""","""Bird""","""Invertebrate""","""Invertebrates""",1.0,1.7,"""On ground""","""Forest""",2.0,"""Dome/Oven""","""Grass""",null,"""Female""",null,"""Poly""","""Various""",24.0,"""Small""",null,"""600""","""1300""",null,null,null
18695,"""Atrichornis clamosus""","""Noisy Scrub-bird""","""Passeriformes""","""Atrichornithidae""","""EN""","""Native""","""Endemic""","""Bird""","""Invertebrate""","""Invertebrates""",1.0,1.3,"""On ground""","""Shrub""",5.0,"""Dome/Oven""","""Shrub/Bush/Vine""",null,"""Female""",null,"""Poly""","""Various""",43.63,"""Small""",null,"""0""","""500""",null,null,null
18866,"""Stipiturus mallee""","""Mallee Emu-wren""","""Passeriformes""","""Maluridae""","""EN""","""Native""","""Endemic""","""Bird""","""Invertebrate""","""Invertebrates""",2.0,1.22,"""On ground""","""Grassland""",3.0,"""Dome/Oven""","""Grass""",null,"""Female""","""Various""",null,"""Sedentary""",6.17,"""Small""",null,"""30""","""500""",null,null,null
18872,"""Amytornis woodwardi""","""White-throated Grasswren""","""Passeriformes""","""Maluridae""","""EN""","""Native""","""Endemic""","""Bird""","""Omnivore""","""Plant""",3.0,1.22,"""On ground""","""Rocky""",2.0,"""Dome/Oven""","""Grass""","""Occasional help""",null,"""Pairs/Family""",null,"""Sedentary""",34.0,"""Small""",null,"""100""","""1000""",null,null,null
19522,"""Dasyornis longirostris""","""Western Bristlebird""","""Passeriformes""","""Dasyornithidae""","""EN""","""Native""","""Endemic""","""Bird""","""Invertebrate""","""Seed""",2.0,1.7,"""On ground""","""Shrub""",1.0,"""Dome/Oven""","""Grass""",null,null,null,"""Mono""","""Various""",34.0,"""Small""",null,"""0""","""50""",null,null,null
19534,"""Pardalotus quadragintus""","""Forty-spotted Pardalote""","""Passeriformes""","""Pardalotidae""","""EN""","""Native""","""Endemic""","""Bird""","""Invertebrate""","""Invertebrates""",2.0,1.7,"""Canopy""","""Forest""",1.0,"""Dome/Oven""","""Stump, Ground, Tree""",null,"""Both""","""Various""","""Mono""","""Various""",11.0,"""Small""",null,"""0""","""1000""",null,null,null
19561,"""Oreoscopus gutturalis""","""Fernwren""","""Passeriformes""","""Acanthizidae""","""EN""","""Native""","""Endemic""","""Bird""","""Invertebrate""","""Invertebrates""",2.0,1.7,"""On ground""","""Forest""",1.0,"""Dome/Oven""","""Stump, Ground, Shrub/Bush/Vine""","""Solitary breeder""","""Female""","""Single/Pairs""",null,"""Various""",18.2,"""Small""",null,"""450""","""1500""",null,null,null
23345,"""Petroica multicolor""","""Norfolk Island Robin""","""Passeriformes""","""Petroicidae""","""EN""","""Native""","""Endemic""","""Bird""","""Invertebrate""","""Invertebrates""",1.0,1.7,"""Mid height""","""Forest""",2.0,"""Cup/Bowl""","""Tree""",null,"""Female""",null,null,"""Sedentary""",15.0,"""Small""",null,"""0""","""320""",null,null,null


## Joining DES data and MS Acccess database 

In [157]:
# Join observations -> header (survey) -> species, with DES-style column names
# so the result can be stacked/joined with the cleaned `des` table later.

# Header lat/lon are DDMM text ("-" where missing); key coords are DDMM numbers.
# Reuses valid_ddmm() and ddmm_to_dd() from the DES cleaning cell.
survey_locations = (
    header
    .select(
        pl.col("survey_id").alias("Survey ID"),
        pl.col("latitude").cast(pl.Int64, strict=False).alias("latitude"),
        pl.col("longitude").cast(pl.Int64, strict=False).alias("longitude"),
        pl.col("k_latitude").cast(pl.Int64).alias("key_latitude"),
        pl.col("k_longitude").cast(pl.Int64).alias("key_longitude"),
        pl.col("locality").alias("Locality"),
        # strict=False turns the odd invalid date (e.g. "1900-01-00") into null
        pl.col("start_date").str.to_date("%Y-%m-%d", strict=False).alias("Observation Start Date"),
        pl.col("end_date").str.to_date("%Y-%m-%d", strict=False).alias("Observation End Date"),
    )
)

# The header has no region code, but every 10-minute grid cell in the location
# table belongs to exactly one region, so look it up by the survey's key cell
cell_regions = (
    locations
    .group_by("bl_Lat", "bl_Lon")
    .agg(pl.col("region_Code").first().alias("Region code"))
)
survey_locations = survey_locations.join(
    cell_regions,
    left_on=["key_latitude", "key_longitude"],
    right_on=["bl_Lat", "bl_Lon"],
    how="left",
)

# Same cleaning and DDMM -> decimal degrees conversion as the DES table
survey_locations = survey_locations.with_columns(
    pl.when(valid_ddmm(c, 9, 45)).then(pl.col(c)).otherwise(None).alias(c)
    for c in ["latitude", "key_latitude"]
).with_columns(
    pl.when(valid_ddmm(c, 112, 160)).then(pl.col(c)).otherwise(None).alias(c)
    for c in ["longitude", "key_longitude"]
).with_columns(
    key_latitude=-ddmm_to_dd("key_latitude"),
    key_longitude=ddmm_to_dd("key_longitude"),
    latitude=-ddmm_to_dd("latitude"),
    longitude=ddmm_to_dd("longitude"),
)

species_names = species.select(
    pl.col("species_id").alias("Species ID"),
    pl.col("species_name").alias("Species Common Name"),
    pl.col("species_sciname").alias("Species Scientific Name"),
)

# Inner join to the header drops the ~1,000 sightings with no survey (no location);
# left join to species keeps the 22 sightings whose species number isn't in the list
atlas_obs = (
    observations
    .select(pl.col("survey_id").alias("Survey ID"), pl.col("species_id").alias("Species ID"))
    .join(survey_locations, on="Survey ID", how="inner")
    .join(species_names, on="Species ID", how="left")
    .select(
        "Survey ID", "Observation Start Date", "Observation End Date",
        "Species ID", "Species Common Name", "Species Scientific Name",
        "latitude", "longitude", "key_latitude", "key_longitude", "Locality", "Region code",
    )
)
atlas_obs

Survey ID,Observation Start Date,Observation End Date,Species ID,Species Common Name,Species Scientific Name,latitude,longitude,key_latitude,key_longitude,Locality,Region code
i64,date,date,i64,str,str,f64,f64,f64,f64,str,str
241376,2015-10-25,2015-10-31,58,"""Purple Swamphen""","""Porphyrio porphyrio""",-31.583333,159.083333,-31.583333,159.083333,"""Lord Howe Island""","""TS"""
241376,2015-10-25,2015-10-31,137,"""Pacific Golden Plover""","""Pluvialis fulva""",-31.583333,159.083333,-31.583333,159.083333,"""Lord Howe Island""","""TS"""
241376,2015-10-25,2015-10-31,129,"""Ruddy Turnstone""","""Arenaria interpres""",-31.583333,159.083333,-31.583333,159.083333,"""Lord Howe Island""","""TS"""
241376,2015-10-25,2015-10-31,153,"""Bar-tailed Godwit""","""Limosa lapponica""",-31.583333,159.083333,-31.583333,159.083333,"""Lord Howe Island""","""TS"""
241376,2015-10-25,2015-10-31,124,"""Black Noddy""","""Anous minutus""",-31.583333,159.083333,-31.583333,159.083333,"""Lord Howe Island""","""TS"""
…,…,…,…,…,…,…,…,…,…,…,…
595767,2001-03-13,2001-03-13,122,"""Common Noddy""","""Anous stolidus""",-31.75,159.25,-31.75,159.25,"""Balls Pyramid""","""TS"""
595767,2001-03-13,2001-03-13,124,"""Black Noddy""","""Anous minutus""",-31.75,159.25,-31.75,159.25,"""Balls Pyramid""","""TS"""
595767,2001-03-13,2001-03-13,922,"""Kermadec Petrel""","""Pterodroma neglecta""",-31.75,159.25,-31.75,159.25,"""Balls Pyramid""","""TS"""


In [158]:
atlas_obs.columns

['Survey ID',
 'Observation Start Date',
 'Observation End Date',
 'Species ID',
 'Species Common Name',
 'Species Scientific Name',
 'latitude',
 'longitude',
 'key_latitude',
 'key_longitude',
 'Locality',
 'Region code']

In [159]:
# Combine the Atlas sightings (up to 2020) with the DES sightings (2020 - 2026).
# DES survey IDs start at 900000 and Atlas IDs stop below 600000, so stacking the
# two tables can't mix up surveys. "Source" records which dataset each row came from.

# Bring the cleaned DES table into the same shape as atlas_obs
des_obs = (
    des
    .with_columns(
        # DES writes missing values as the text "NULL"
        pl.col("Locality", "Region code").replace("NULL", None),
        pl.col("Species ID").cast(pl.Int64, strict=False),
        pl.col("Observation Start Date", "Observation End Date").cast(pl.Date),
    )
    # DES has common names but no scientific names, so take those from the species list
    .join(species_names.select("Species ID", "Species Scientific Name"), on="Species ID", how="left")
    .select(atlas_obs.columns)
)

all_obs = pl.concat([
    atlas_obs.with_columns(Source=pl.lit("Atlas")),
    des_obs.with_columns(Source=pl.lit("DES")),
])
all_obs

Survey ID,Observation Start Date,Observation End Date,Species ID,Species Common Name,Species Scientific Name,latitude,longitude,key_latitude,key_longitude,Locality,Region code,Source
i64,date,date,i64,str,str,f64,f64,f64,f64,str,str,str
241376,2015-10-25,2015-10-31,58,"""Purple Swamphen""","""Porphyrio porphyrio""",-31.583333,159.083333,-31.583333,159.083333,"""Lord Howe Island""","""TS""","""Atlas"""
241376,2015-10-25,2015-10-31,137,"""Pacific Golden Plover""","""Pluvialis fulva""",-31.583333,159.083333,-31.583333,159.083333,"""Lord Howe Island""","""TS""","""Atlas"""
241376,2015-10-25,2015-10-31,129,"""Ruddy Turnstone""","""Arenaria interpres""",-31.583333,159.083333,-31.583333,159.083333,"""Lord Howe Island""","""TS""","""Atlas"""
241376,2015-10-25,2015-10-31,153,"""Bar-tailed Godwit""","""Limosa lapponica""",-31.583333,159.083333,-31.583333,159.083333,"""Lord Howe Island""","""TS""","""Atlas"""
241376,2015-10-25,2015-10-31,124,"""Black Noddy""","""Anous minutus""",-31.583333,159.083333,-31.583333,159.083333,"""Lord Howe Island""","""TS""","""Atlas"""
…,…,…,…,…,…,…,…,…,…,…,…,…
902577,2022-08-21,2022-08-21,9,"""Stubble Quail""","""Coturnix pectoralis""",-31.916667,149.416667,-31.916667,149.416667,"""Moggymill""","""WS""","""DES"""
902577,2022-08-21,2022-08-21,705,"""Australian Magpie""","""Gymnorhina tibicen""",-31.916667,149.416667,-31.916667,149.416667,"""Moggymill""","""WS""","""DES"""
902577,2022-08-21,2022-08-21,364,"""Willie Wagtail""","""Rhipidura leucophrys""",-31.916667,149.416667,-31.916667,149.416667,"""Moggymill""","""WS""","""DES"""


In [160]:
all_obs.select(
    pl.col("Observation End Date").min().alias("min"),
    pl.col("Observation End Date").max().alias("max"),
)


min,max
date,date
1900-01-01,2028-02-28


In [161]:
all_obs.columns 

['Survey ID',
 'Observation Start Date',
 'Observation End Date',
 'Species ID',
 'Species Common Name',
 'Species Scientific Name',
 'latitude',
 'longitude',
 'key_latitude',
 'key_longitude',
 'Locality',
 'Region code',
 'Source']

In [162]:
traits.columns

['ID',
 'Scientific_Name',
 'Common_Name',
 'Order_Name',
 'Family_Name',
 'ALA_IUCN',
 'Local_Status',
 'Population',
 'Species_Type',
 'Primary_Diet',
 'ALA_Diet_Code',
 'Diet_Breadth',
 'ESI',
 'Foraging_Strategy',
 'Primary_Habitat',
 'Habitat_Breadth',
 'Nest_Type',
 'Nest_Substrate',
 'Cooperative_Breeder',
 'Incubating_Sex',
 'Social_Behaviour',
 'Monogamy',
 'Mobility',
 'Body_Mass_g',
 'Body_Size',
 'Exclusion_Zone',
 'Elev_NormMin',
 'Elev_NormMax',
 'breeding_season_start',
 'breeding_season_end',
 'breeding_season_notes']

# Slide 1 

In [163]:
import plotly.graph_objects as go

# Reporting rate of a species in a decade = share of that decade's surveys that recorded it.
# Atlas surveys only, 1960s-2010s: effort is thin before 1960, and DES (2020+) is a different program.
DECADES = list(range(1960, 2020, 10))

# Most early "surveys" are one-off sightings of a single species (84% in the 1960s vs 16% in the 2010s),
# which would make every species look like it's increasing. Keeping only surveys that list at least
# 20 species (i.e. full checklists) holds the median list at 36-45 species in every decade.
MIN_LIST_LENGTH = 20

atlas_decades = (
    all_obs
    .filter(pl.col("Source") == "Atlas")
    .with_columns(decade=pl.col("Observation End Date").dt.year() // 10 * 10)
    .filter(pl.col("decade").is_in(DECADES))
    .filter(pl.col("Species ID").n_unique().over("Survey ID") >= MIN_LIST_LENGTH)
)
surveys_per_decade = atlas_decades.group_by("decade").agg(surveys=pl.col("Survey ID").n_unique())
detections = atlas_decades.group_by("Species ID", "decade").agg(detections=pl.col("Survey ID").n_unique())

# Species list -> traits: IOC_species_id is text like "31.0", traits ID is a float
species_traits = (
    species
    .select(
        pl.col("species_id").alias("Species ID"),
        pl.col("IOC_species_id").cast(pl.Float64, strict=False).cast(pl.Int64).alias("ID"),
    )
    .join(traits.with_columns(pl.col("ID").cast(pl.Int64)), on="ID", how="inner")
)

# Every recorded species x every decade, so a decade where a species wasn't seen counts as 0
reporting_rates = (
    detections.select("Species ID").unique()
    .join(pl.DataFrame({"decade": DECADES}, schema={"decade": pl.Int32}), how="cross")
    .join(detections, on=["Species ID", "decade"], how="left")
    .join(surveys_per_decade, on="decade")
    .with_columns(reporting_rate=pl.col("detections").fill_null(0) / pl.col("surveys") * 100)
    .join(species_traits, on="Species ID", how="inner")
    # Extinct species sit at 0 throughout and only flatten their groups
    .filter(pl.col("Local_Status") != "Extinct")
)

TRAITS = {
    "Primary_Diet": "Diet",
    "Primary_Habitat": "Habitat",
    "Foraging_Strategy": "Foraging strategy",
    "Body_Size": "Body size",
    "Mobility": "Mobility",
    "ALA_IUCN": "IUCN status",
}
MAX_GROUPS = 8          # the palette has 8 hues; smaller groups fold into "Other"
MIN_GROUP_SPECIES = 3   # a mean over 1-2 species is too noisy to read a trend from
COLOURS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
OTHER_COLOUR = "#8a8985"  # grey, so the catch-all doesn't read as a real group

def trait_lines(trait: str) -> pl.DataFrame:
    """Mean reporting rate per decade for each group of a trait."""
    rates = reporting_rates.filter(pl.col(trait).is_not_null())
    group_sizes = (
        rates.group_by(trait).agg(pl.col("Species ID").n_unique().alias("n"))
        .filter(pl.col("n") >= MIN_GROUP_SPECIES)
        .sort("n", descending=True)
    )
    keep = group_sizes[trait].head(MAX_GROUPS - 1 if group_sizes.height > MAX_GROUPS else MAX_GROUPS)
    rates = rates.with_columns(
        pl.when(pl.col(trait).is_in(keep.implode())).then(pl.col(trait)).otherwise(pl.lit("Other")).alias(trait)
    )
    return (
        rates
        .group_by(pl.col(trait).alias("group"), "decade")
        .agg(rate=pl.col("reporting_rate").mean(), species=pl.col("Species ID").n_unique())
        .sort("decade")
    )

fig = go.Figure()
buttons = []
for t, (trait, label) in enumerate(TRAITS.items()):
    lines = trait_lines(trait)
    # Order groups by their latest rate so the legend reads top-to-bottom like the line ends
    order = lines.filter(pl.col("decade") == max(DECADES)).sort("rate", descending=True)["group"].to_list()
    order = [g for g in order if g != "Other"] + (["Other"] if "Other" in order else [])
    for i, group in enumerate(order):
        line = lines.filter(pl.col("group") == group)
        fig.add_trace(go.Scatter(
            x=[f"{d}s" for d in line["decade"]],
            y=line["rate"].to_list(),
            name=group,
                        mode="lines+markers",
            line=dict(width=2, color=OTHER_COLOUR if group == "Other" else COLOURS[i]),
            marker=dict(size=8),
            customdata=line["species"].to_list(),
            hovertemplate=f"{group}: %{{y:.1f}}% (%{{customdata}} species)<extra></extra>",
            visible=(t == 0),
        ))
    buttons.append((label, len(order)))

# One dropdown entry per trait: show only that trait's lines
traces_so_far = 0
menu = []
for label, n in buttons:
    visible = [False] * len(fig.data)
    visible[traces_so_far:traces_so_far + n] = [True] * n
    traces_so_far += n
    menu.append(dict(label=label, method="update",
                     args=[{"visible": visible},
                           {"title.text": f"Mean reporting rate of NSW bird species by {label.lower()}, 1960s–2010s"}]))

fig.update_layout(
    template="plotly_white",
    title=dict(text=f"Mean reporting rate of NSW bird species by {buttons[0][0].lower()}, 1960s–2010s",
               x=0, y=0.97, yanchor="top"),
    # Trait picker sits under the title, above the plot
    updatemenus=[dict(buttons=menu, x=0, xanchor="left", y=1.04, yanchor="bottom", showactive=True)],
    xaxis=dict(title="Decade", showgrid=False),
    yaxis=dict(title="Mean reporting rate (% of surveys)", rangemode="tozero", gridcolor="#e6e5e1"),
    hovermode="x unified",
    legend=dict(title="Group"),
    margin=dict(t=120),
    height=550,
)
fig.show()

In [164]:
sd_id = observations["survey_id"]
sd_id.n_unique()

573479

# Slide 2

In [165]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

# Survey coverage by decade: number of surveys in each 10-minute grid cell, one map per decade.
# Every panel shares one log colour scale, so a colour means the same count on every map.
# Grey cells were surveyed in some decade but not this one, which shows where coverage was missing.
COVERAGE_DECADES = list(range(1960, 2020, 10))
CELL = 10 / 60  # grid cells are 10 minutes wide
# Cell centres sit at 5, 15, 25... minutes, so a grid with edges on whole degrees puts each
# centre in the middle of a cell. (Starting at NSW_LON[0] = 140°30' put centres exactly on the
# edges, and rounding scattered them between columns, leaving empty stripes.)
GRID_LAT0, GRID_LON0 = np.floor(NSW_LAT[0]), np.floor(NSW_LON[0])

survey_cells = (
    header
    .select(
        "survey_id",
        pl.col("end_date").str.to_date("%Y-%m-%d", strict=False).dt.year().alias("year"),
        (-ddmm_to_dd("k_latitude")).alias("lat"),
        ddmm_to_dd("k_longitude").alias("lon"),
    )
    .with_columns(decade=pl.col("year") // 10 * 10)
    .filter(
        pl.col("decade").is_in(COVERAGE_DECADES),
        pl.col("lat").is_between(*NSW_LAT),
        pl.col("lon").is_between(*NSW_LON),
    )
    # Whole-number row/column of each cell on a regular grid, so the maps line up exactly
    .with_columns(
        row=((pl.col("lat") - GRID_LAT0) / CELL).floor().cast(pl.Int64),
        col=((pl.col("lon") - GRID_LON0) / CELL).floor().cast(pl.Int64),
    )
)

cell_counts = survey_cells.group_by("decade", "row", "col").agg(surveys=pl.len())
decade_totals = (
    survey_cells.group_by("decade")
    .agg(surveys=pl.len(), cells=pl.struct("row", "col").n_unique())
    .sort("decade")
)

n_rows = cell_counts["row"].max() + 1
n_cols = cell_counts["col"].max() + 1
lats = GRID_LAT0 + (np.arange(n_rows) + 0.5) * CELL
lons = GRID_LON0 + (np.arange(n_cols) + 0.5) * CELL

# Cells surveyed in any decade: the grey backdrop on every panel
ever = np.full((n_rows, n_cols), np.nan)
everywhere = cell_counts.select("row", "col").unique()
ever[everywhere["row"].to_numpy(), everywhere["col"].to_numpy()] = 1

# Log colour scale: 1 survey to the busiest cell of any decade
max_log = np.log10(cell_counts["surveys"].max())
tick_values = list(range(int(max_log) + 1))
tick_labels = [f"{10**v:,}" for v in tick_values]

titles = [
    f"<b>{row['decade']}s</b><br>{row['surveys']:,} surveys · {row['cells']:,} cells"
    for row in decade_totals.iter_rows(named=True)
]
fig = make_subplots(rows=2, cols=3, subplot_titles=titles,
                    horizontal_spacing=0.02, vertical_spacing=0.1)

for i, decade in enumerate(COVERAGE_DECADES):
    r, c = i // 3 + 1, i % 3 + 1
    counts = cell_counts.filter(pl.col("decade") == decade)
    grid = np.full((n_rows, n_cols), np.nan)
    grid[counts["row"].to_numpy(), counts["col"].to_numpy()] = np.log10(counts["surveys"].to_numpy())

    fig.add_trace(go.Heatmap(x=lons, y=lats, z=ever, colorscale=[[0, "#e6e5e1"], [1, "#e6e5e1"]],
                             showscale=False, hoverinfo="skip"), row=r, col=c)
    fig.add_trace(go.Heatmap(x=lons, y=lats, z=grid, coloraxis="coloraxis", hoverinfo="skip"),
                  row=r, col=c)

# Keep the maps' shape true: at NSW's latitude a degree of longitude is ~0.84 of a degree of latitude.
# The x range stops at the coast, so Lord Howe Island (~159°E) is off the map.
for i in range(len(COVERAGE_DECADES)):
    suffix = "" if i == 0 else str(i + 1)
    fig.layout[f"xaxis{suffix}"].update(range=[140.8, 153.8], visible=False)
    fig.layout[f"yaxis{suffix}"].update(range=list(NSW_LAT), visible=False,
                                        scaleanchor=f"x{suffix}", scaleratio=1 / np.cos(np.radians(33)))

fig.update_layout(
    template="plotly_white",
    title=dict(text="Atlas survey coverage of NSW by decade, 1960s–2010s", x=0),
    coloraxis=dict(
        colorscale="Plasma_r", cmin=0, cmax=max_log,
        colorbar=dict(title="Surveys<br>per cell", tickvals=tick_values, ticktext=tick_labels),
    ),
    height=700, width=1200,
    margin=dict(l=10, r=10, t=90, b=10),
)
fig.show()

# Slide 3

# Slide 4